# Difference Drivers — NEW vs OLD, train, all three bureaus

Multivariate two-sample test (`zaml DifferenceDrivers`): train a small
XGBoost (10 trees, depth 3) to predict whether a row came from the OLD or
the NEW processed data. Two readings:

- **separation AUC** — 0.5 means the datasets are indistinguishable; since
  ~94% of applicants are bit-identical between OLD and NEW, the all-rows AUC
  is expected to sit barely above 0.5, and that IS the result ("nearly
  indistinguishable"). The changed-rows-only run is the sharp version.
- **feature importance** — which features the classifier uses to tell NEW
  from OLD = the multivariate difference drivers, interactions included
  (what univariate PSI can't see).

What to expect per bureau: equifax and transunion should be driven purely by
`percent_*` features (denominator fix only); experian should also surface
`number_*` features — the placeholder change's footprint.

NEW for experian = `new_normalized_and_processed/experian_train/processed`
(placeholder change included); eq/tu = `processed_new` (behavior-identical
on the branch). OLD = `processed_old`. Model-engine kernel.

In [1]:
import numpy as np
import pandas as pd
from zaml.analyze.data_analysis.difference_drivers import DifferenceDrivers

DATA    = '/home/jag/payment-processor-research/payment_processing_research_data'
BUREAUS = ['equifax', 'experian', 'transunion']
SAMPLE_N = 100_000   # per side for the all-rows run (keeps the SHAP pass fast)
SEED     = 0

def processed_dir(variant, bureau):
    if variant == 'new' and bureau == 'experian':
        return f'{DATA}/new_normalized_and_processed/experian_train/processed'
    return f'{DATA}/samples/{bureau}_train/processed_{variant}'

/home/jag/.conda/envs/model_engine_2_py310/lib/python3.10/site-packages/zaml/common/utils/io.py:17: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [2]:
# feature list: number + percent 'in_last' features (same filters as Testing.ipynb)
import pyarrow.parquet as pq
import glob

schema_file = sorted(glob.glob(f'{DATA}/samples/transunion_train/processed_new/part-*.parquet'))[0]
cols = pq.read_schema(schema_file).names
percent_features = [c for c in cols if 'percent_of_DQ' in c and 'in_last' in c.lower()]
number_features  = [c for c in cols if 'in_last' in c.lower() and c not in percent_features]
features = percent_features + number_features
print(f'{len(percent_features)} percent + {len(number_features)} number/other = {len(features)} features')

1143 percent + 1637 number/other = 2780 features


In [3]:
def run_difference_drivers(bureau, top=15):
    new = pd.read_parquet(processed_dir('new', bureau), columns=features)
    old = pd.read_parquet(processed_dir('old', bureau), columns=features)
    idx = new.index.intersection(old.index)
    new, old = new.loc[idx], old.loc[idx]
    print(f'\n########## {bureau}: {len(idx):,} common applicants ##########')

    # which applicants changed AT ALL across these features
    changed = np.zeros(len(idx), dtype=bool)
    for c in features:
        changed |= ~np.isclose(new[c].astype('float64'), old[c].astype('float64'), equal_nan=True)
    print(f'changed on >=1 feature: {changed.sum():,} ({changed.mean():.2%})')

    results = {}
    # 1) all rows (subsampled for speed)
    rng  = np.random.RandomState(SEED)
    take = rng.choice(len(idx), size=min(SAMPLE_N, len(idx)), replace=False)
    dd = DifferenceDrivers()
    stats = dd.calculate(old.iloc[take], new.iloc[take])
    print(f'\nALL rows (n={len(take):,}/side): separation AUC = {dd.auc:.4f}')
    print(stats.sort_values('importance', ascending=False).head(top).round(4).to_string())
    results['all'] = (dd.auc, stats)

    # 2) changed rows only -- the sharp version
    if changed.sum() >= 1000:
        dd_c = DifferenceDrivers()
        stats_c = dd_c.calculate(old.loc[changed], new.loc[changed])
        print(f'\nCHANGED rows only (n={changed.sum():,}/side): separation AUC = {dd_c.auc:.4f}')
        print(stats_c.sort_values('importance', ascending=False).head(top).round(4).to_string())
        results['changed'] = (dd_c.auc, stats_c)
    else:
        print('\n(too few changed rows for the changed-only run)')

    del new, old
    return results

In [4]:
results_equifax = run_difference_drivers('equifax')


########## equifax: 394,321 common applicants ##########
changed on >=1 feature: 39,532 (10.03%)


ntree_limit is deprecated, use `iteration_range` or model slicing instead.
ntree_limit is deprecated, use `iteration_range` or model slicing instead.
ntree_limit is deprecated, use `iteration_range` or model slicing instead.
ntree_limit is deprecated, use `iteration_range` or model slicing instead.
ntree_limit is deprecated, use `iteration_range` or model slicing instead.
ntree_limit is deprecated, use `iteration_range` or model slicing instead.
ntree_limit is deprecated, use `iteration_range` or model slicing instead.
ntree_limit is deprecated, use `iteration_range` or model slicing instead.
ntree_limit is deprecated, use `iteration_range` or model slicing instead.
ntree_limit is deprecated, use `iteration_range` or model slicing instead.
ntree_limit is deprecated, use `iteration_range` or model slicing instead.
ntree_limit is deprecated, use `iteration_range` or model slicing instead.
ntree_limit is deprecated, use `iteration_range` or model slicing instead.
ntree_limit is deprecated


ALL rows (n=100,000/side): separation AUC = 0.5398
                                                                                                importance
trade_max_percent_of_DQ30_in_last_12_months__recently_opened_open_accounts                          0.1814
trade_mean_percent_of_DQ30_in_last_12_months__recently_opened_open_accounts                         0.1436
trade_max_percent_of_DQ30_in_last_24_months__unsecure_open_accounts                                 0.1290
trade_max_percent_of_DQ30_in_last_24_months__active_open_accounts                                   0.1259
trade_mean_percent_of_DQ30_or_greater_in_last_6_months__derog_open_charge_card                      0.1157
trade_mean_percent_of_DQ30_or_greater_in_last_6_months__recently_opened_open_accounts               0.0640
trade_mean_number_of_DQ30_in_last_12_months__unsecure_open_accounts                                 0.0516
trade_max_percent_of_DQ30_in_last_24_months__all_open_accounts                              

ntree_limit is deprecated, use `iteration_range` or model slicing instead.
ntree_limit is deprecated, use `iteration_range` or model slicing instead.
ntree_limit is deprecated, use `iteration_range` or model slicing instead.
ntree_limit is deprecated, use `iteration_range` or model slicing instead.
ntree_limit is deprecated, use `iteration_range` or model slicing instead.
ntree_limit is deprecated, use `iteration_range` or model slicing instead.
ntree_limit is deprecated, use `iteration_range` or model slicing instead.
ntree_limit is deprecated, use `iteration_range` or model slicing instead.
ntree_limit is deprecated, use `iteration_range` or model slicing instead.
ntree_limit is deprecated, use `iteration_range` or model slicing instead.
ntree_limit is deprecated, use `iteration_range` or model slicing instead.
ntree_limit is deprecated, use `iteration_range` or model slicing instead.
ntree_limit is deprecated, use `iteration_range` or model slicing instead.
ntree_limit is deprecated


CHANGED rows only (n=39,532/side): separation AUC = 0.8823
                                                                                      importance
trade_max_percent_of_DQ30_in_last_24_months__active_open_accounts                         0.2208
trade_max_percent_of_DQ30_in_last_24_months__all_open_accounts                            0.2183
trade_max_number_of_DQ30_in_last_24_months__all_open_accounts                             0.1309
trade_mean_percent_of_DQ30_or_greater_in_last_6_months__derog_open_charge_card            0.1154
trade_max_number_of_DQ30_in_last_24_months__active_open_accounts                          0.0939
trade_max_percent_of_DQ30_in_last_24_months__all_open_charge_card                         0.0671
trade_mean_percent_of_DQ30_in_last_24_months__all_open_accounts                           0.0394
trade_max_percent_of_DQ30_in_last_24_months__unsecure_open_accounts                       0.0363
trade_mean_percent_of_DQ30_in_last_12_months__all_open_accounts    

ntree_limit is deprecated, use `iteration_range` or model slicing instead.


In [14]:
results_equifax['changed'][1].sort_values(by = 'importance', ascending = False)

,importance
trade_max_percent_of_DQ30_in_last_24_months__active_open_accounts,0.220835
trade_max_percent_of_DQ30_in_last_24_months__all_open_accounts,0.218288
trade_max_number_of_DQ30_in_last_24_months__all_open_accounts,0.130856
trade_mean_percent_of_DQ30_or_greater_in_last_6_months__derog_open_charge_card,0.115405
trade_max_number_of_DQ30_in_last_24_months__active_open_accounts,0.093924
...,...
trade_mean_percent_of_DQ30_in_last_12_months__recently_opened_open_unsecure_personal,0.000000
trade_min_percent_of_DQ30_in_last_12_months__recently_opened_open_unsecure_personal,0.000000
trade_max_percent_of_DQ30_in_last_12_months__recently_opened_open_unsecure_personal,0.000000
trade_mean_percent_of_DQ60_in_last_12_months__recently_opened_open_unsecure_personal,0.000000


In [5]:
results_experian = run_difference_drivers('experian')


########## experian: 394,241 common applicants ##########
changed on >=1 feature: 39,998 (10.15%)


ntree_limit is deprecated, use `iteration_range` or model slicing instead.
ntree_limit is deprecated, use `iteration_range` or model slicing instead.
ntree_limit is deprecated, use `iteration_range` or model slicing instead.
ntree_limit is deprecated, use `iteration_range` or model slicing instead.
ntree_limit is deprecated, use `iteration_range` or model slicing instead.
ntree_limit is deprecated, use `iteration_range` or model slicing instead.
ntree_limit is deprecated, use `iteration_range` or model slicing instead.
ntree_limit is deprecated, use `iteration_range` or model slicing instead.
ntree_limit is deprecated, use `iteration_range` or model slicing instead.
ntree_limit is deprecated, use `iteration_range` or model slicing instead.
ntree_limit is deprecated, use `iteration_range` or model slicing instead.
ntree_limit is deprecated, use `iteration_range` or model slicing instead.
ntree_limit is deprecated, use `iteration_range` or model slicing instead.
ntree_limit is deprecated


ALL rows (n=100,000/side): separation AUC = 0.5291
                                                                                       importance
trade_max_percent_of_DQ30_in_last_12_months__recently_opened_open_accounts                 0.2358
trade_mean_percent_of_DQ30_or_greater_in_last_6_months__recently_opened_open_accounts      0.1664
trade_mean_percent_of_DQ30_in_last_12_months__recently_opened_open_accounts                0.1539
trade_max_percent_of_DQ30_in_last_24_months__active_open_accounts                          0.0922
trade_max_percent_of_DQ30_in_last_24_months__unsecure_open_accounts                        0.0772
trade_max_percent_of_DQ30_in_last_24_months__non_derog_open_accounts                       0.0653
trade_max_percent_of_DQ30_in_last_24_months__non_derog_open_revolving                      0.0520
trade_max_percent_of_DQ30_in_last_24_months__individual_open_accounts                      0.0500
trade_max_percent_of_DQ30_in_last_24_months__all_open_accounts    

ntree_limit is deprecated, use `iteration_range` or model slicing instead.
ntree_limit is deprecated, use `iteration_range` or model slicing instead.
ntree_limit is deprecated, use `iteration_range` or model slicing instead.
ntree_limit is deprecated, use `iteration_range` or model slicing instead.
ntree_limit is deprecated, use `iteration_range` or model slicing instead.
ntree_limit is deprecated, use `iteration_range` or model slicing instead.
ntree_limit is deprecated, use `iteration_range` or model slicing instead.
ntree_limit is deprecated, use `iteration_range` or model slicing instead.
ntree_limit is deprecated, use `iteration_range` or model slicing instead.
ntree_limit is deprecated, use `iteration_range` or model slicing instead.
ntree_limit is deprecated, use `iteration_range` or model slicing instead.
ntree_limit is deprecated, use `iteration_range` or model slicing instead.
ntree_limit is deprecated, use `iteration_range` or model slicing instead.
ntree_limit is deprecated


CHANGED rows only (n=39,998/side): separation AUC = 0.8375
                                                                                       importance
trade_max_percent_of_DQ30_in_last_24_months__active_open_accounts                          0.3859
trade_max_number_of_DQ30_in_last_24_months__active_open_accounts                           0.2067
trade_mean_percent_of_DQ30_in_last_24_months__active_open_accounts                         0.0933
trade_max_percent_of_DQ30_in_last_24_months__unsecure_open_accounts                        0.0670
trade_percent_accounts_with_DQ30_or_greater_in_last_12_months__unsecure_open_accounts      0.0389
trade_mean_percent_of_DQ30_in_last_12_months__active_open_accounts                         0.0375
trade_max_percent_of_DQ30_in_last_24_months__non_derog_open_accounts                       0.0277
trade_max_number_of_DQ30_in_last_24_months__unsecure_open_accounts                         0.0270
trade_percent_accounts_with_DQ30_or_greater_in_last_24_mon

ntree_limit is deprecated, use `iteration_range` or model slicing instead.


In [6]:
results_transunion = run_difference_drivers('transunion')


########## transunion: 395,913 common applicants ##########
changed on >=1 feature: 39,730 (10.04%)


ntree_limit is deprecated, use `iteration_range` or model slicing instead.
ntree_limit is deprecated, use `iteration_range` or model slicing instead.
ntree_limit is deprecated, use `iteration_range` or model slicing instead.
ntree_limit is deprecated, use `iteration_range` or model slicing instead.
ntree_limit is deprecated, use `iteration_range` or model slicing instead.
ntree_limit is deprecated, use `iteration_range` or model slicing instead.
ntree_limit is deprecated, use `iteration_range` or model slicing instead.
ntree_limit is deprecated, use `iteration_range` or model slicing instead.
ntree_limit is deprecated, use `iteration_range` or model slicing instead.
ntree_limit is deprecated, use `iteration_range` or model slicing instead.
ntree_limit is deprecated, use `iteration_range` or model slicing instead.
ntree_limit is deprecated, use `iteration_range` or model slicing instead.
ntree_limit is deprecated, use `iteration_range` or model slicing instead.
ntree_limit is deprecated


ALL rows (n=100,000/side): separation AUC = 0.5486
                                                                                        importance
trade_mean_percent_of_DQ30_or_greater_in_last_6_months__derog_open_charge_card              0.2783
trade_max_percent_of_DQ30_or_greater_in_last_6_months__active_open_charge_card              0.1408
trade_min_percent_of_DQ30_or_greater_in_last_6_months__derog_open_charge_card               0.1342
trade_percent_accounts_with_DQ120_or_greater_in_last_6_months__all_open_charge_card         0.1145
trade_max_percent_of_DQ30_in_last_24_months__unsecure_open_accounts                         0.1044
trade_percent_accounts_with_DQ120_or_greater_in_last_6_months__active_open_charge_card      0.0829
trade_max_percent_of_DQ30_in_last_12_months__recently_opened_open_accounts                  0.0563
trade_max_percent_of_DQ30_in_last_24_months__active_open_accounts                           0.0269
trade_min_percent_of_DQ60_in_last_12_months__derog_open_c

ntree_limit is deprecated, use `iteration_range` or model slicing instead.
ntree_limit is deprecated, use `iteration_range` or model slicing instead.
ntree_limit is deprecated, use `iteration_range` or model slicing instead.
ntree_limit is deprecated, use `iteration_range` or model slicing instead.
ntree_limit is deprecated, use `iteration_range` or model slicing instead.
ntree_limit is deprecated, use `iteration_range` or model slicing instead.
ntree_limit is deprecated, use `iteration_range` or model slicing instead.
ntree_limit is deprecated, use `iteration_range` or model slicing instead.
ntree_limit is deprecated, use `iteration_range` or model slicing instead.
ntree_limit is deprecated, use `iteration_range` or model slicing instead.
ntree_limit is deprecated, use `iteration_range` or model slicing instead.
ntree_limit is deprecated, use `iteration_range` or model slicing instead.
ntree_limit is deprecated, use `iteration_range` or model slicing instead.
ntree_limit is deprecated


CHANGED rows only (n=39,730/side): separation AUC = 0.9348
                                                                                        importance
trade_mean_percent_of_DQ30_or_greater_in_last_6_months__derog_open_charge_card              0.2728
trade_max_percent_of_DQ30_in_last_24_months__active_open_accounts                           0.2109
trade_max_percent_of_DQ30_in_last_24_months__all_open_accounts                              0.1680
trade_max_number_of_DQ30_in_last_24_months__active_open_accounts                            0.0921
trade_max_number_of_DQ30_in_last_24_months__all_open_accounts                               0.0665
trade_percent_accounts_with_DQ120_or_greater_in_last_6_months__all_open_charge_card         0.0450
trade_max_percent_of_DQ30_or_greater_in_last_6_months__active_open_charge_card              0.0432
trade_max_percent_of_DQ30_in_last_24_months__all_open_charge_card                           0.0259
trade_percent_accounts_with_DQ120_or_greater_in_l

ntree_limit is deprecated, use `iteration_range` or model slicing instead.


In [7]:
# summary: top-10 drivers per bureau side by side (changed-rows run)
summary = {}
for b, res in [('equifax', results_equifax), ('experian', results_experian),
               ('transunion', results_transunion)]:
    if 'changed' in res:
        auc, stats = res['changed']
        top10 = stats.sort_values('importance', ascending=False).head(10)
        summary[b] = pd.Series([f'{i+1}. {c} ({v:.3f})' for i, (c, v) in
                                enumerate(top10['importance'].items())])
        print(f'{b}: changed-rows separation AUC = {auc:.4f}')
pd.DataFrame(summary)

equifax: changed-rows separation AUC = 0.8823
experian: changed-rows separation AUC = 0.8375
transunion: changed-rows separation AUC = 0.9348


,equifax,experian,transunion
0,1. trade_max_percent_of_DQ30_in_last_24_months...,1. trade_max_percent_of_DQ30_in_last_24_months...,1. trade_mean_percent_of_DQ30_or_greater_in_la...
1,2. trade_max_percent_of_DQ30_in_last_24_months...,2. trade_max_number_of_DQ30_in_last_24_months_...,2. trade_max_percent_of_DQ30_in_last_24_months...
2,3. trade_max_number_of_DQ30_in_last_24_months_...,3. trade_mean_percent_of_DQ30_in_last_24_month...,3. trade_max_percent_of_DQ30_in_last_24_months...
3,4. trade_mean_percent_of_DQ30_or_greater_in_la...,4. trade_max_percent_of_DQ30_in_last_24_months...,4. trade_max_number_of_DQ30_in_last_24_months_...
4,5. trade_max_number_of_DQ30_in_last_24_months_...,5. trade_percent_accounts_with_DQ30_or_greater...,5. trade_max_number_of_DQ30_in_last_24_months_...
5,6. trade_max_percent_of_DQ30_in_last_24_months...,6. trade_mean_percent_of_DQ30_in_last_12_month...,6. trade_percent_accounts_with_DQ120_or_greate...
6,7. trade_mean_percent_of_DQ30_in_last_24_month...,7. trade_max_percent_of_DQ30_in_last_24_months...,7. trade_max_percent_of_DQ30_or_greater_in_las...
7,8. trade_max_percent_of_DQ30_in_last_24_months...,8. trade_max_number_of_DQ30_in_last_24_months_...,8. trade_max_percent_of_DQ30_in_last_24_months...
8,9. trade_mean_percent_of_DQ30_in_last_12_month...,9. trade_percent_accounts_with_DQ30_or_greater...,9. trade_percent_accounts_with_DQ120_or_greate...
9,10. trade_mean_percent_of_DQ30_in_last_12_mont...,10. trade_mean_percent_of_DQ30_in_last_12_mont...,10. trade_min_percent_of_DQ30_or_greater_in_la...
